## Install dependencies

In [ ]:
# %%capture
# %%bash
# VENV_DIR=".adaptive_search_venv"

# # Create virtual environment if it doesn't exist
# if [ ! -d "$VENV_DIR" ]; then
#     python3 -m venv "$VENV_DIR"
#     echo "Virtual environment created at: $VENV_DIR"
# fi

# # Determine pip executable path
# if [[ "$OSTYPE" == "msys" || "$OSTYPE" == "win32" ]]; then
#     PIP_EXEC="$VENV_DIR/Scripts/pip.exe"
# else
#     PIP_EXEC="$VENV_DIR/bin/pip"
# fi

# Upgrade pip and install dependencies
% install --upgrade pip
%pip install \
    langchain \
    langgraph \
    langchain_google_vertexai \
    matplotlib \
    seaborn \
    pandas \
    numpy \
    python-dotenv \
    ipykernel \
    aiohttp \
    pip-system-certs

# echo "Dependencies installed in the virtual environment."

### Activate virtual environmentt
- Restart notebook or select the newly installed kernel in Jupyter

## Read env variables
You need to have `ADAPTIVE_SEARCH_HOST` and `ADAPTIVE_SEARCH_TOKEN` defined in `.env`

In [ ]:
# Environment setup: load .env so AdaptiveSearchAgent + tools can read credentials
from dotenv import load_dotenv, find_dotenv
import os

# Load environment variables
load_dotenv(find_dotenv())

In [ ]:
ADAPTIVE_SEARCH_HOST = os.getenv("ADAPTIVE_SEARCH_HOST")
print('Using adaptive search host:', ADAPTIVE_SEARCH_HOST)

if 'ADAPTIVE_SEARCH_TOKEN' not in os.environ:
    raise ValueError("ADAPTIVE_SEARCH_TOKEN environment variable is not set.")

## Adaptive Search tools
These functions fetch data from the Adaptive Search API. Used as a `tool` by the LangGraph agent.

In [ ]:
import os
import logging
import aiohttp
from langchain_core.tools import tool

# Set up logging
logger = logging.getLogger(__name__)

async def get_adaptive_search_access_token() -> str:
    """Get access token from Okta for adaptive search service."""
    ADAPTIVE_SEARCH_HOST = os.getenv("ADAPTIVE_SEARCH_HOST")
    refresh_url = f"{ADAPTIVE_SEARCH_HOST}/oauth2/refresh"
    logger.info("REFRESH URL: %s", refresh_url)

    async with aiohttp.ClientSession() as session:
        async with session.post(
            refresh_url,
            json={"refresh_token": os.getenv('ADAPTIVE_SEARCH_TOKEN')},
            timeout=aiohttp.ClientTimeout(total=100),
        ) as refresh_response:
            logger.info("ACCESS TOKEN TAKEN SUCCESS %s", await refresh_response.json())
            refresh_response.raise_for_status()
            return await refresh_response.json()


# --- Tool Definition ---
async def get_adaptive_search_response(query: str) -> dict:
    logger.info("Getting adaptive search access token")
    token = await get_adaptive_search_access_token()
    ADAPTIVE_SEARCH_HOST = os.getenv("ADAPTIVE_SEARCH_HOST")
    logger.info("Adaptive Search host: %s", ADAPTIVE_SEARCH_HOST)
    data = {"query": query}
    logger.info("Making request to adaptive search service: %s", f"{ADAPTIVE_SEARCH_HOST}/api/v2/search")

    async with aiohttp.ClientSession() as session:
        async with session.post(
            url=f"{ADAPTIVE_SEARCH_HOST}/api/v2/search",
            json=data,
            timeout=aiohttp.ClientTimeout(total=100),
            headers = {
                "Authorization": f"Bearer {token}",
                "Content-Type": "application/json",
                }
        ) as adaptive_search_response:
            logger.info("Received adaptive search response with status code: %s", adaptive_search_response.status)
            adaptive_search_response.raise_for_status()
            return await adaptive_search_response.json()

@tool(parse_docstring=True)
async def get_adaptive_search_data(query: str) -> str:
    """
    <purpose>
    Retrieve comprehensive financial and corporate data from premium S&P Global datasets to answer questions about companies, markets, executives, transactions, and financial metrics.
    </purpose>

    <data_coverage>
    This tool provides access to extensive financial and corporate datasets including:

    **Financial Data:**
    - Complete financial statements (income statements, balance sheets, cash flow)
    - Revenue, profit, margins, and all financial line items
    - Quarterly and annual financial data with historical trends
    - Segment-level financials by geography and business unit
    - Financial ratios and multiples (P/E, EV/EBITDA, debt-to-equity, etc.)
    - Market capitalization and enterprise value data

    **Market & Trading Data:**
    - Daily stock prices and historical price movements
    - Stock performance analysis and trends
    - Market valuation metrics

    **Corporate Information:**
    - Company competitors and competitive analysis
    - Product portfolios and business descriptions
    - Executive and board member profiles
    - Executive compensation details and trends

    **Corporate Actions & Transactions:**
    - Mergers and acquisitions activity
    - Stock buyback programs and status
    - Private placements and debt issuances
    - Bankruptcy filings and restructuring events

    **Communications & Disclosures:**
    - Earnings call transcripts and management commentary
    - SEC filings and regulatory documents
    - Corporate risk disclosures and material information
    - Business news articles and key corporate announcements
    - Industry trends and sector analysis

    All data is sourced from S&P Global's validated databases and professional-grade financial platforms.
    </data_coverage>

    <capabilities>
    This tool excels at handling complex, compound queries and can search across multiple companies, metrics, timeframes, and data sources simultaneously in a single call. Always prefer comprehensive compound queries over multiple separate calls.

    **Optimal Usage:**
    - Send complex multi-part questions as single queries rather than breaking them down
    - Include multiple companies, metrics, or timeframes in one search to maximize efficiency
    - Combine related financial concepts in a single query for comprehensive results
    - Only split queries if they involve completely unrelated topics that cannot be logically combined
    </capabilities>

    <examples>
    - "What are Apple, Microsoft, and Google's revenue growth rates and profit margins for Q3 2024?"
    - "Compare Nvidia's data center revenue vs gaming revenue over the past 4 quarters and their forward guidance"
    - "Show me the earnings results, stock performance, and analyst reactions for the top 5 tech companies this quarter"
    - "Compare Tesla, Ford, and GM revenue, margins, and guidance for the last 3 quarters"
    - "Who are the executives at Amazon and what were their 2023 compensation packages?"
    - "What acquisitions has Microsoft made since 2020 and what was discussed about integration on earnings calls?"
    - "What climate-related risks does BP disclose and how do their margins compare to other oil companies?"
    </examples>

    Args:
        query: Complete financial question that can include multiple companies, metrics, timeframes, and related concepts across financial statements, market data, corporate actions, executive information, or business communications
    """
    logger.info("Tool called: get_adaptive_search_data with query: %s", query)
    response = await get_adaptive_search_response(query=query)
    grounded_data = response.get("response")
    logger.info("Received grounded_data with %s items", len(grounded_data) if grounded_data else 0)
    output = []
    if grounded_data is None:
        logger.warning("No grounded data found in response")
        return "No data found for the query."
    for data in grounded_data:
        sources = []
        for source in data["sources"]:
            if source.get("source") and source["source"].get("uri"):
                source_data = {
                    "name": source["source"]["name"],
                    "link": source["source"]["uri"],
                    "description": f"Provider: {source['source']['provider']}",
                    "notes": [],
                    "limitations": [],
                }
                sources.append(source_data)
        item_data = {
            "data": data.get("data"),
            "source": sources
        }
        output.append(item_data)
    return '\n'.join(str(item) for item in output) if output else "No data found for the query."

## Plotting tools
Another `tool` for LangGraph. If user asks to visualize something - this will be invoked by the model.

In [ ]:
import base64
from datetime import datetime
import os
import re
import tempfile
from typing import Any

from langchain_core.tools import tool
from pydantic import BaseModel


class Base64PNGImage(BaseModel):
    file_name: str
    file_path: str
    file_bytes: str

    def __hash__(self) -> int:
        return hash(self.file_path)

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Base64PNGImage):
            return False
        return self.file_path == other.file_path


def _generate_unique_filename(filename: str) -> str:
    """Generate a unique filename based on timestamp and provided filename."""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    # Sanitize filename
    sanitized_name = re.sub(r"[^a-zA-Z0-9_-]", "_", filename.lower().strip())
    sanitized_name = re.sub(
        r"_{2,}", "_", sanitized_name
    )  # Replace multiple underscores with single
    sanitized_name = sanitized_name[:50]  # Limit length
    # Remove .png extension if present and add timestamp
    if sanitized_name.endswith(".png"):
        sanitized_name = sanitized_name[:-4]
    return f"{sanitized_name}_{timestamp}.png"


def _ensure_plots_directory() -> str:
    """Ensure the plots directory exists and return its absolute path."""
    plots_dir = os.getenv("IMAGE_STORAGE_DIR", "./generated_plots")
    # Convert to absolute path to avoid issues when changing working directory
    plots_dir = os.path.abspath(plots_dir)
    os.makedirs(plots_dir, exist_ok=True)
    return plots_dir


def _execute_plot_code(python_code: str, filename: str) -> tuple[str, Base64PNGImage | None]:
    """Execute Python plotting code in a sandboxed environment and save to permanent location.

    This function provides a secure execution environment for user-generated Python plotting code.
    It creates a temporary working directory, executes the provided code with access to plotting
    libraries, and moves the resulting plot to a permanent storage location.

    Args:
        python_code: Complete Python code that generates a plot and saves it as 'plot.png'
        filename: Desired filename for the plot (without extension)

    Returns:
        tuple[str, Base64PNGImage | None]: A tuple containing:
            - str: Success message with file path or detailed error information
            - Base64PNGImage | None: Encoded image object for LangChain integration, or None if error occurred

        LangChain/LangGraph Integration:
            This function returns a tuple to integrate with LangChain's tool calling system. When a tool
            is marked with response_format="content_and_artifact", the resulting ToolMessage will have
            whatever is in the second tuple position attached as an artifact to the tool message. This
            allows the plotting tool to both provide text feedback create image artifacts for processing

    Security Features:
        - Executes code in isolated temporary directory
        - Restricted namespace with only plotting libraries
        - Automatic cleanup of temporary files
        - Original working directory restoration

    Expected Code Structure:
        The provided code should create data, generate visualization, and save using:
        plt.savefig('plot.png')
    """
    # Import required libraries first
    import matplotlib

    matplotlib.use("Agg")  # Use non-interactive backend
    import datetime
    import math
    import random

    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import seaborn as sns

    # Create a restricted namespace with plotting libraries
    restricted_globals: dict[str, Any] = {
        # Pre-import all libraries into the namespace
        "plt": plt,
        "sns": sns,
        "pd": pd,
        "np": np,
        "datetime": datetime,
        "math": math,
        "random": random,
        "matplotlib": matplotlib,
    }

    try:
        # Create permanent plots directory and generate unique filename
        plots_dir = _ensure_plots_directory()
        unique_filename = _generate_unique_filename(filename)
        final_plot_path = os.path.join(plots_dir, unique_filename)

        # Create a temporary directory for execution
        with tempfile.TemporaryDirectory() as temp_dir:
            # Change to temp directory for code execution
            original_cwd = os.getcwd()
            os.chdir(temp_dir)

            try:
                # Execute the user's code
                exec(python_code, restricted_globals)

                # Check if plot.png was created
                temp_plot_path = os.path.join(temp_dir, "plot.png")
                if not os.path.exists(temp_plot_path):
                    return (
                        "Error: Code executed successfully but no 'plot.png' file was created. "
                        "Please ensure your code saves the plot using plt.savefig('plot.png').",
                        None,
                    )

                # Move the plot to permanent location
                import shutil

                shutil.copy2(temp_plot_path, final_plot_path)

                # Get file size for reporting
                file_size = os.path.getsize(final_plot_path)

                # Read the PNG file and create Base64PNGImage object
                try:
                    with open(final_plot_path, "rb") as f:
                        file_bytes = base64.b64encode(f.read()).decode()
                        image_obj = Base64PNGImage(
                            file_name=unique_filename,
                            file_path=final_plot_path,
                            file_bytes=file_bytes,
                        )
                except Exception as e:
                    return (
                        f"Plot generated and saved to {final_plot_path} but failed to encode as base64: {str(e)}",
                        None,
                    )

                message = (
                    f"Plot generated successfully and saved to {final_plot_path} "
                    f"({file_size} bytes). You can reference this image in your response "
                    f"using markdown: ![{filename}]({final_plot_path})"
                )

                return (message, image_obj)

            finally:
                # Restore original working directory
                os.chdir(original_cwd)
                # Clean up any remaining plots
                plt.close("all")

    except ImportError as e:
        missing_package = str(e).split("'")[1] if "'" in str(e) else str(e)
        return (
            f"ImportError: Required package '{missing_package}' is not available. "
            f"Please check that all necessary plotting libraries are installed. "
            f"Available packages: matplotlib, seaborn, pandas, numpy.",
            None,
        )
    except SyntaxError as e:
        return (
            f"SyntaxError in provided code: {str(e)} (Line {e.lineno}). "
            f"Please check your Python syntax and ensure proper indentation.",
            None,
        )
    except NameError as e:
        return (
            f"NameError: {str(e)}. The variable or function is not defined in the available namespace. "
            f"Available libraries: plt (matplotlib.pyplot), sns (seaborn), pd (pandas), np (numpy), "
            f"datetime, math, random. Make sure to create all data within your code.",
            None,
        )
    except Exception as e:
        error_type = type(e).__name__
        return (
            f"{error_type}: {repr(e)}. "
            f"Please check your code for errors. Common issues: "
            f"1) Data not properly defined, 2) Incorrect function calls, "
            f"3) Missing plt.savefig('plot.png'), 4) Incompatible data types.",
            None,
        )


@tool(parse_docstring=True, response_format="content_and_artifact")
def generate_plot(python_code: str, filename: str) -> tuple[str, Base64PNGImage | None]:
    """Generate a data visualization by executing Python plotting code in a controlled environment.

    <purpose>
    Execute user-provided Python code to create charts, plots, and data visualizations. The code runs in a
    sandboxed environment with access to essential plotting and data manipulation libraries. The generated
    plot is saved as an image and returned as base64-encoded data for display.
    </purpose>

    <available_packages>
    The execution environment provides access to the following packages with standard aliases:

    **Core Data & Plotting Libraries:**
    - `matplotlib.pyplot` (as `plt`) - Primary plotting library
    - `seaborn` (as `sns`) - Statistical data visualization
    - `pandas` (as `pd`) - Data manipulation and analysis
    - `numpy` (as `np`) - Numerical computing

    **Additional Utilities:**
    - `datetime` - Date and time handling
    - `math` - Mathematical functions
    - `random` - Random number generation
    - Standard Python built-ins (list, dict, range, etc.)

    These packages are pre-imported and ready to use with their standard aliases.
    </available_packages>

    <plotting_guidelines>
    **Visual Design Standards:**
    - Use professional color palettes: seaborn default, 'viridis', 'plasma', 'Set2', or matplotlib's 'tab10'
    - Set figure size appropriately: plt.figure(figsize=(10, 6)) for most charts
    - Include clear titles, axis labels, and legends when relevant
    - Use grid lines sparingly: plt.grid(True, alpha=0.3) for subtle gridlines
    - Ensure text is readable with appropriate font sizes (12+ for labels, 14+ for titles)

    **Chart Type Recommendations:**
    - Line plots: Time series data, trends over time
    - Bar charts: Categorical comparisons, rankings
    - Scatter plots: Correlation analysis, relationship exploration
    - Histograms: Distribution analysis
    - Box plots: Statistical summaries, outlier detection
    - Heatmaps: Correlation matrices, 2D data density

    **Code Structure Requirements:**
    - Create all data within the code (no external file dependencies)
    - Use matplotlib.pyplot.savefig() to save the plot as PNG
    - Call plt.close() after saving to free memory
    - Include plt.tight_layout() before saving for proper spacing
    </plotting_guidelines>

    <data_creation_examples>
    **Create sample data directly in code:**
    ```python
    # Time series data
    dates = pd.date_range('2024-01-01', periods=12, freq='M')
    values = [100, 105, 98, 110, 115, 108, 120, 125, 118, 130, 135, 142]

    # Categorical data
    categories = ['A', 'B', 'C', 'D', 'E']
    values = [23, 45, 56, 78, 32]

    # Random data for distributions
    data = np.random.normal(100, 15, 1000)

    # DataFrame creation
    df = pd.DataFrame({
        'x': range(10),
        'y': [i**2 for i in range(10)],
        'category': ['Group A']*5 + ['Group B']*5
    })
    ```
    </data_creation_examples>

    <execution_requirements>
    **Mandatory Code Structure:**
    1. Create or define all necessary data within the code
    2. Generate the visualization using available libraries
    3. Save the plot: `plt.savefig('plot.png', dpi=150, bbox_inches='tight')`
    4. Clean up: `plt.close()`

    **File Naming:**
    - Always save plots as 'plot.png' in the current directory
    - Use high DPI (150+) for crisp image quality
    - Include bbox_inches='tight' to prevent clipping

    **🚨 CRITICAL: Python Code String Formatting Requirements**
    When calling this tool, the python_code parameter MUST be formatted correctly:

    **✅ CORRECT - Use triple quotes for multi-line strings:**
    ```python
    python_code = \"\"\"
    import pandas as pd
    import matplotlib.pyplot as plt

    data = [['227.94', 'USD', '2025-08-25'], ['228.84', 'USD', '2025-08-22']]
    df = pd.DataFrame(data, columns=['price', 'currency', 'date'])

    plt.figure(figsize=(10, 6))
    plt.plot(df['date'], df['price'])
    plt.title('Stock Price Trend')
    plt.savefig('plot.png', dpi=150, bbox_inches='tight')
    plt.close()
    \"\"\"
    ```

    **❌ INCORRECT - Do NOT use escaped newlines:**
    ```python
    python_code = \"\\nimport pandas as pd\\nplt.plot()\\nplt.savefig('plot.png')\\n\"
    ```

    **Key Points:**
    - Use actual newlines, not escaped `\\n` sequences
    - Format the code exactly as you would write it in a Python file
    - Proper indentation and spacing are preserved with triple quotes
    - This ensures the code executes correctly in the sandboxed environment
    </execution_requirements>

    Args:
        python_code (str): Complete Python code that creates data, generates a plot, and saves it as 'plot.png'.
            Code should be self-contained with no external dependencies or file reads.
        filename (str): Desired filename for the saved plot (without .png extension). Will be sanitized and
            made unique with timestamp.

    Returns:
        tuple[str, Base64PNGImage | None]: A tuple containing:
            - Success message with file path or detailed error information. The success message
              includes the file path where the plot was saved and suggested markdown for referencing it.
            - Base64PNGImage object containing the encoded image data, or None if an error occurred.
    """
    return _execute_plot_code(python_code, filename)


## System prompt
The main system prompt of the ReAct agent.

In [ ]:
"""
System prompt for the S&P Global Adaptive Search Agent
"""

SYSTEM_PROMPT = """
<role>
You are the S&P Global Adaptive Search Agent, a specialized financial research assistant equipped with a powerful AI search tool called "Adaptive Search" that can search across multiple financial datasets. Your goal is to help users find accurate financial information efficiently.
</role>

<identity_and_introduction>
**Agent Identity**: You are the S&P Global Adaptive Search Agent powered by Kensho with access to comprehensive S&P Global financial datasets.

**Current Date Context**: Today's date is {today_string}. Use this as your reference point for "current", "recent", "latest", and relative time expressions (e.g., "this quarter", "last quarter", "YTD").

**When asked about your identity or to introduce yourself, respond with:**
"I'm the S&P Global Adaptive Search Agent powered by Kensho. I have access to comprehensive financial and corporate data from S&P Global's premium datasets, including financial statements, earnings data, executive information, market data, and corporate communications. I can help you research companies, analyze financial metrics, compare competitors, and access detailed business intelligence across public companies."

**Keep introductions concise** - provide a high-level overview without extensive detail about specific capabilities or data sources unless specifically requested.
</identity_and_introduction>

<conversation_context>
**🚫 ABSOLUTE RULE: CLARIFICATION IS BANNED - MINE CONVERSATION HISTORY INSTEAD**

You are ABSOLUTELY FORBIDDEN from asking users to clarify, specify, or provide more information. Context is ALWAYS available in conversation history.

**MANDATORY CONTEXT SOURCES:**
- **Previous responses you gave**: Companies, competitors, data you just provided
- **Previous user messages**: What they asked about before
- **Tool results**: All data and company names from recent searches
- **Conversation flow**: The logical continuation of the discussion

**CONTEXT EXTRACTION RULES:**
- **"their/they/those companies"** = ALL companies mentioned in your last response
- **"the leadership teams"** = Executive teams of companies from recent context
- **"these competitors"** = Competitor companies you just listed
- **"same data/analysis"** = Apply previous analysis type to new context
- **Vague pronouns** = Default to the most recently discussed companies/topics

**CONVERSATION HISTORY MINING PROCESS:**
1. **Scan your last response** for company names, competitors, data mentioned
2. **Extract ALL entities** from recent discussion (companies, people, metrics)
3. **Use ALL companies mentioned** in the last 2-3 exchanges
4. **Default to comprehensive searches** covering all relevant entities
5. **Never narrow scope** - broader searches are always better

**CONTEXT USAGE EXAMPLES:**
- Previous: Listed 10 S&P Global competitors → "their leadership teams" = Leadership teams of ALL 10 competitors
- Previous: Analyzed Apple, Microsoft, Google → "what about margins?" = Margins for Apple, Microsoft, Google
- Previous: Tesla earnings data → "show competitors" = Tesla competitors + their earnings data
- After listing competitors → "their compensation" = Compensation for ALL listed competitors
- User says "them/they/those" → ALL companies from recent context

**🚫 BANNED PHRASES - NEVER SAY:**
- "Could you clarify which companies..."
- "Which specific companies are you referring to..."
- "I need more information about..."
- "Could you specify which..."
- "To provide accurate information, I need..."

**✅ REQUIRED BEHAVIOR:**
- **NEVER ask "which companies"** - extract from conversation history
- **NEVER ask "what timeframe"** - use recent/relevant periods
- **NEVER ask "which metrics"** - search comprehensively
- **ALWAYS search first** - explanations come after providing data
- Extract company names from conversation history
- Use ALL companies mentioned in recent context
- Search broadly if any ambiguity exists
</conversation_context>

<tool_capabilities>
**Adaptive Search Tool:**
- Performs AI-powered searches across financial datasets including earnings calls, financial statements, stock prices, analyst reports, and market data
- Can handle multiple parallel queries in a single call (e.g., searching for data on multiple companies simultaneously)
- Limited to ONE search pass per tool call - cannot perform follow-up searches or chain queries within a single call

**Plot Generation Tool:**
- Creates professional data visualizations from financial data using Python plotting libraries
- Executes custom Python code to generate charts, graphs, and visual analytics
- Supports matplotlib, seaborn, pandas, and numpy for comprehensive data visualization
- Saves high-quality PNG images to disk and returns the file path for referencing
- You must include the description parameter when calling generate_plot for meaningful filenames
- After the tool runs, include the generated image in your response using the markdown provided by the tool
- Ideal for showing trends, comparisons, distributions, and relationships in financial data
</tool_capabilities>

<query_strategy>
**AGGRESSIVE QUERYING - GO BIG OR GO HOME**: The Adaptive Search tool can handle massive, complex queries. Your default approach should be to create the most comprehensive, ambitious queries possible. Never hold back.

**MANDATORY COMPREHENSIVE APPROACH**:
- **Always expand simple questions** into multi-dimensional searches
- **Include related companies, metrics, and timeframes** even if not explicitly requested
- **Cast the widest possible net** - search for more data than requested
- **Use parallel queries liberally** - multiple broad searches are better than one narrow search
- **Assume the user wants comprehensive analysis** - go beyond the minimum

**AGGRESSIVE EXPANSION RULES**:
- Single company → Add competitors and industry context
- One metric → Add related financial metrics and ratios
- One timeframe → Add historical trends and comparisons
- Vague request → Search multiple interpretations simultaneously
- Simple question → Transform into comprehensive analysis request

<aggressive_expansion_examples>
- "What was Tesla's revenue?" → EXPAND TO: "Tesla's revenue, profit margins, guidance, competitor comparison (Ford, GM), EV market trends, and analyst reactions for the last 4 quarters"
- "Apple earnings" → EXPAND TO: "Apple's latest earnings results, segment performance, guidance, competitor analysis vs Microsoft and Google, analyst reactions, and stock performance"
- "Tech stocks" → EXPAND TO: "Performance analysis of major tech companies (Apple, Microsoft, Google, Amazon, Meta, Tesla, Nvidia), sector trends, earnings highlights, and market outlook"
</aggressive_expansion_examples>

**PARALLEL QUERY STRATEGY - USE MULTIPLE SEARCHES**:
- When unsure about scope → Run 2-3 parallel broad searches
- When dealing with ambiguity → Search all reasonable interpretations
- When facing complex requests → Break into multiple comprehensive parallel queries
- Default to multiple searches rather than asking for clarification

**NEVER LIMIT YOURSELF** - The tool can handle:
- Queries covering dozens of companies simultaneously
- Multi-year historical analysis with projections
- Cross-industry comparisons and sector analysis
- Complex financial calculations and ratio analysis
</query_strategy>

<best_practices>
1. **ASSUME MAXIMUM SCOPE**: If 10 competitors mentioned, search ALL 10, not a subset
2. **EXPAND AMBIGUOUS REQUESTS**: Turn unclear questions into comprehensive multi-company searches
3. **BROAD OVER NARROW**: When unsure of scope, search more companies/data, not fewer
4. **CONTEXTUAL CONTINUITY**: Each response builds on ALL information from previous responses
5. **PARALLEL COMPREHENSIVE SEARCHES**: Multiple broad searches beat single narrow ones
6. **SEARCH IMMEDIATELY**: No explanations before searching - get data first
7. **PROACTIVE VISUALIZATION**: Generate plots for numerical data to enhance understanding and analysis
</best_practices>

<visualization_strategy>
**WHEN TO CREATE PLOTS - BE PROACTIVE:**

**🎯 ALWAYS GENERATE VISUALIZATIONS FOR:**
- **Time series data**: Revenue trends, stock prices, quarterly performance over multiple periods
- **Comparisons**: Multiple companies' metrics side-by-side (revenue, margins, growth rates)
- **Rankings**: Top performers, market leaders, competitive positioning
- **Distributions**: Salary ranges, valuation multiples, performance spreads
- **Relationships**: Correlation between metrics (revenue vs margins, P/E vs growth)
- **Trend analysis**: Growth trajectories, seasonal patterns, historical comparisons

**📊 OPTIMAL CHART TYPES FOR FINANCIAL DATA:**
- **Line charts**: Time series (stock prices, revenue over quarters, growth trends)
- **Bar charts**: Company comparisons (revenue by company, market share, rankings)
- **Grouped bar charts**: Multi-metric comparisons (revenue vs profit by company)
- **Scatter plots**: Relationship analysis (P/E vs growth rate, risk vs return)
- **Heatmaps**: Correlation matrices, performance across sectors/time periods
- **Box plots**: Distribution analysis (executive compensation, valuation ranges)

**🚀 PROACTIVE PLOTTING RULES:**
1. **Default to visualization**: If data is numerical and multi-dimensional, create a plot
2. **Enhance every analysis**: Text + tables + visualization = comprehensive response
3. **Multiple perspectives**: Create 2-3 different chart types for complex datasets
4. **Tell a story**: Use plots to highlight key insights and trends
5. **Professional quality**: Always include titles, labels, legends, and proper formatting

**📈 PLOT GENERATION WORKFLOW:**
1. After getting adaptive search data, immediately assess if visualization would add value
2. Extract relevant numerical data from the adaptive search results
3. Create Python code that builds the data structures and generates professional plots
4. Call generate_plot with both python_code and description parameters (e.g., generate_plot(code, "revenue_trends"))
5. The tool will save the image and return a message with the file path and suggested markdown
6. Copy the markdown from the tool's response and include it in your final response to display the image

CRITICAL PYTHON CODE FORMATTING RULES:
- ALWAYS use triple quotes for multi-line Python code strings
- NEVER use escaped newlines (\\n) in Python code strings
- Generate clean, properly indented Python code with actual newlines
- Format code as you would write it normally, not as escaped strings
- Example CORRECT format:
  python_code = \"\"\"
  import pandas as pd
  import matplotlib.pyplot as plt

  data = [['value1', 'value2'], ['value3', 'value4']]
  df = pd.DataFrame(data)
  plt.plot(df[0], df[1])
  plt.savefig('plot.png')
  plt.close()
  \"\"\"
**💡 EXAMPLE SCENARIOS REQUIRING PLOTS:**
- "Show me Apple's revenue" → Line chart of quarterly revenue trends + comparison bar chart with competitors
- "Compare tech companies" → Multi-metric grouped bar chart + scatter plot of growth vs margins
- "Executive compensation at banks" → Box plot distribution + bar chart of top executives
- "Stock performance analysis" → Line chart of price trends + comparison with market indices
- "Industry analysis" → Heatmap of sector performance + trend lines for key metrics

**⚠️ VISUALIZATION REQUIREMENTS:**
- NEVER skip plotting opportunities for numerical data
- ALWAYS create self-contained Python code with embedded data
- INCLUDE proper titles, axis labels, and professional styling
- USE appropriate color schemes and readable fonts
- ALWAYS provide a descriptive name when calling generate_plot (e.g., "revenue_comparison", "stock_performance")
- COPY the exact markdown from the tool's response to display the image in your final response
- The tool saves images automatically - you just need to include the markdown it provides
</visualization_strategy>

<response_format>
**CRITICAL RESPONSE FORMATTING REQUIREMENT:**
- Your response MUST be a single, complete markdown string
- ALL content including text, tables, images, and formatting should be in ONE continuous markdown response
- DO NOT use separate tool outputs or multiple response parts
- Structure your response as a cohesive markdown document

**Content Guidelines:**
- Provide clear, well-structured answers based on the search results
- Display structured data in markdown format with more rows than columns, putting quarter or time frame as columns when appropriate
- Reference previous conversation points when relevant to show continuity
- When including images, use the exact markdown provided by the plot tool within your response

**CRITICAL CITATION REQUIREMENTS - ABSOLUTELY MANDATORY:**

🚨 CITATION ENFORCEMENT: You MUST ALWAYS include a "Citations" section at the bottom of EVERY SINGLE RESPONSE that uses the adaptive search tool. This is ABSOLUTELY REQUIRED and NON-NEGOTIABLE under ALL circumstances.

**MULTI-TURN CONVERSATION ALERT:**
Even in follow-up questions or subsequent messages in a conversation, you MUST include citations if you use any adaptive search data. Do NOT assume citations from previous messages carry over. Each response that uses adaptive search data requires its own citations section.

**ZERO TOLERANCE POLICY:**
- NEVER skip citations, even for simple queries
- NEVER assume citations are optional
- NEVER omit citations because you think the user already knows the sources
- NEVER exclude citations in follow-up responses
- EVERY adaptive search tool call REQUIRES citations in that specific response

**Citation Format:**
- Create a "## Citations" header at the end of your response
- List all sources as bullet points where:
  - The bullet text should be an informative, unique description
  - The underlying link should be the URL (from the 'link' field)
  - Format: `- [Source description](URL)`

**INTELLIGENT CITATION NAMING - REQUIRED PROCESS:**

**Step 1: Evaluate Available Names**
- Check if 'description' field provides specific, unique information
- Check if 'name' field is informative and distinct from other sources
- Identify if names are vague (e.g., "Financial Report", "Company Data", "Document") or repetitive

**Step 2: Generate Enhanced Names When Needed**
When description/name is vague, repetitive, or uninformative, analyze the URL to create unique identifiers:
- **Document IDs**: Extract document/report IDs from URLs (e.g., "10-K Filing #0001234567")
- **Date specificity**: Add specific dates found in URLs (e.g., "Q3 2024 Earnings Report")
- **Company specificity**: Include company ticker/name from URL path (e.g., "Apple Inc. SEC Filing")
- **Report types**: Identify specific document types (e.g., "Form 10-Q", "Proxy Statement", "8-K Filing")
- **Source platform**: Include data provider when relevant (e.g., "Bloomberg Terminal Data", "S&P Capital IQ Report")

**Step 3: Ensure Uniqueness**
- Each citation must have a distinct, informative name
- Avoid generic names like "Document 1", "Report 2", "Financial Data"
- Include specific identifiers that help users understand what each source contains

**Enhanced Citation Examples:**
```
## Citations
- [Apple Inc. Form 10-Q Q3 2024 Filing #001-36743](https://sec.gov/filings/apple-10q-20240930)
- [Microsoft Earnings Call Transcript - October 24, 2024](https://investor.microsoft.com/earnings-transcript-q1-2025)
- [S&P Capital IQ Executive Compensation Database - Tech Sector](https://platform.capitaliq.com/exec-comp/tech-q3-2024)
- [Tesla Inc. Proxy Statement DEF 14A #001-34756](https://sec.gov/proxy-statements/tesla-2024-proxy)
```

**Citation Rules:**
- Include ALL sources returned by the adaptive search tool, even if not directly quoted
- Generate unique, informative names using URL analysis when standard names are vague
- If a source has no link, display the enhanced description without making it a hyperlink
- Sources should be ordered by relevance to the main content
- This applies to EVERY response, including 1st, 2nd, 3rd, and all subsequent messages in a conversation
- **Special URL handling**: When the only URL you have is www.capitaliq.spglobal.com or something very similar, make the name of that source "S&P Capital IQ Pro"
</response_format>

Remember: Your goal is to get the most comprehensive and accurate financial information with the minimum number of tool calls, while maintaining awareness of the full conversation context. NEVER REFUSE - ALWAYS SEARCH FIRST.

🚨🚨🚨 FINAL BEHAVIORAL CHECKLIST 🚨🚨🚨
BEFORE RESPONDING TO ANY USER QUESTION:
✅ Did I scan conversation history for companies/entities to use?
✅ Did I extract ALL relevant companies from previous responses?
✅ Did I search immediately without asking for clarification?
✅ Did I use maximum scope (all companies mentioned, not a subset)?
✅ Did I attempt an adaptive search (unless inappropriate/harmful content)?
✅ Did I expand the query to be comprehensive rather than minimal?
✅ Did I assess if the data would benefit from visualization?
✅ Did I create plots for numerical/comparative data when appropriate?
✅ Does my response end with "## Citations"?
✅ Are ALL sources from the adaptive search tool listed?

IF ANY ANSWER IS NO → FIX IMMEDIATELY BEFORE RESPONDING

🚫 **NEVER SAY THESE BANNED PHRASES:**
- "Could you clarify which companies you're referring to?"
- "I need more information about which companies..."
- "To provide accurate information, could you specify..."
- "Which specific companies would you like me to analyze?"
- "Could you provide more details about..."

✅ **ALWAYS DO THIS INSTEAD:**
- Extract companies from conversation history
- Use ALL entities from recent context
- Search broadly and comprehensively
- Provide data first, explanations second
"""

## Post model hooks
These are invoked after LLM generated the answer to format the answer as Markdown text.

In [ ]:
import logging
from typing import Annotated
import operator

from langchain_core.messages import ToolMessage, AIMessage
from langgraph.prebuilt.chat_agent_executor import AgentStatePydantic

logger = logging.getLogger(__name__)


class PngImages(BaseModel):
    """Container for images generated during one invocation"""

    images: list[Base64PNGImage]


class GroundingAgentState(AgentStatePydantic):
    # a list of PngImages means one PngImages container per invocation turn
    # should be an empty list when no images are generated in that invocation
    generated_images: Annotated[list[PngImages], operator.add] = []


def _scrape_new_images(state: GroundingAgentState) -> list[Base64PNGImage]:
    """Extract new images from the state that haven't been seen before.

    Args:
        state: The current agent state containing messages and previously generated images

    Returns:
        List of new Base64PNGImage objects found in recent tool messages
    """
    # get set of previously seen images
    prev_images = set()
    for png_images_obj in state.generated_images:
        prev_images.update(png_images_obj.images)

    # get new images
    new_images = []
    for message in state.messages:
        if isinstance(message, ToolMessage) and isinstance(message.artifact, Base64PNGImage):
            image = message.artifact
            if image not in prev_images:
                new_images.append(image)

    return new_images


def post_model_hook(state: GroundingAgentState) -> dict:
    """Post-model hook for the LangGraph pre-built React agent.

    This is a node in the graph that is inserted after the agent node. Its purpose is to only run on the
    terminal state, meaning that the agent is finished processing the request. It updates the agent state's
    generated_images key with any new image artifacts generated during that invocation.

    The result is that downstream of the agent execution, the application can fetch the latest set of
    generated images from the last element in the generated_images list.

    Args:
        state (GroundingAgentState): The current agent state containing messages and previously generated images.

    Returns:
        dict: State update dictionary containing new generated images if this is a terminal state,
              or an empty dict if the agent is still processing.
    """

    last_msg = state.messages[-1]
    is_terminal = isinstance(last_msg, AIMessage) and not last_msg.tool_calls

    if not is_terminal:
        return {}

    logger.info(
        "Post-model hook: terminal state detected, about to scrape images from message history"
    )

    # get new images using the extracted function
    new_images = _scrape_new_images(state)

    logger.info(
        "Post-model hook: scraped %d new images with filenames: %s",
        len(new_images),
        [img.file_name for img in new_images],
    )

    out = dict()

    # return new images or an empty list so that generated_images[-1] access is always valid
    out["generated_images"] = [PngImages(images=new_images)]

    return out

## Create agent
`project` is the GCP project that provided Gemini access. Can be swapped for another LLM accessor.

In [ ]:
from datetime import datetime
from langgraph.prebuilt import create_react_agent
from langchain_google_vertexai import ChatVertexAI
from langgraph.checkpoint.memory import MemorySaver
from enum import Enum

tools = [get_adaptive_search_data, generate_plot]

class VertexAIModel(Enum):
    GEMINI_2_5_PRO = "gemini-2.5-pro"
    GEMINI_2_5_FLASH = "gemini-2.5-flash"

model = ChatVertexAI(
    model=VertexAIModel.GEMINI_2_5_FLASH.value,
    project="kensho-adaptive-search",
    temperature=0.0
)

today_string = datetime.now().strftime('%B %d, %Y')
formatted_prompt = SYSTEM_PROMPT.format(today_string=today_string)

checkpointer = MemorySaver()

react_graph = create_react_agent(
    model,
    tools=tools,
    checkpointer=checkpointer,
    prompt=formatted_prompt,
    post_model_hook=post_model_hook,
    # debug=True,
)

from IPython.display import Image, display
display(Image(react_graph.get_graph().draw_mermaid_png()))

## Invocation Utils

In [ ]:
def reset_conversation():
    checkpointer.delete_thread('notebook-demo')

## Example 1, text response

In [ ]:
from IPython.display import Markdown, display
import nest_asyncio
nest_asyncio.apply()  # Allow nested event loops in Jupyter

# Example query
query = "When did Microsoft last mention 'AI growth' in an earnings call, and what was its closing stock price that day?"

reset_conversation()  # Reset conversation for a fresh start
result = await react_graph.ainvoke({"messages": [("user", query)]}, config={"configurable": {"thread_id": "notebook-demo"}})

# Display the final AI message
final_msg = result["messages"][-1]
display(Markdown(final_msg.content))

## Example 2, making a plot

In [ ]:
from IPython.display import Markdown, display

query = "Plot Amazon stock price YTD"

reset_conversation()  # Reset conversation for a fresh start
result = await react_graph.ainvoke({"messages": [("user", query)]}, config={"configurable": {"thread_id": "notebook-demo"}})

# Display the final AI message
final_msg = result["messages"][-1]
display(Markdown(final_msg.content))

### Notebook-specific workaround to show the image in Github

Note that the below code is ONLY for rendering the image in Github for presentation purposes. This is needed because the reference to the local image file breaks when the notebook is pushed to github. Users of this notebook do NOT need to use this code in their agent.

In [ ]:
# Display with base64 encoded image for GitHub compatibility
import re
import base64

content = final_msg.content
# Find image path and replace with base64
image_match = re.search(r'!\[(.*?)\]\((.*?)\)', content)
if image_match:
    alt_text, image_path = image_match.groups()
    with open(image_path, "rb") as f:
        img_b64 = base64.b64encode(f.read()).decode()
    content = re.sub(r'!\[(.*?)\]\((.*?)\)', f'<img src="data:image/png;base64,{img_b64}" alt="{alt_text}">', content)

display(Markdown(content))